<a href="https://colab.research.google.com/github/CopingMoa/ForenXAI-v2/blob/main/ebm_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install pandas numpy scikit-learn interpret optuna gdown joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 21.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 14.5 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl size=4

In [ ]:

import os
import json
import joblib
import gdown
import numpy as np
import pandas as pd
import optuna

from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score, roc_auc_score, average_precision_score

# ============================================================
# CONFIG
# ============================================================
RANDOM_STATE = 42
LABEL_COLUMN = "label_binary"
DRIVE_URL = "https://drive.google.com/drive/u/1/folders/1dz9X40Xuz5--c2n7FCR3rSlLclDU763c"
DATA_DIR = "raw_data"
OUTPUT_DIR = "ebm_models"
TEST_SIZE = 0.20
CV_SPLITS = 5
OPTUNA_TRIALS = 30

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. ACQUIRE DATASET
# ============================================================
gdown.download_folder(
    url=DRIVE_URL,
    output=DATA_DIR,
    quiet=False
)

csv_files = []
for root, _, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, file))

if not csv_files:
    raise FileNotFoundError("No CSV file found.")

if len(csv_files) == 1:
    CSV_PATH = csv_files[0]
else:
    frames = [pd.read_csv(f, low_memory=False) for f in csv_files]
    df = pd.concat(frames, ignore_index=True)
    CSV_PATH = os.path.join(DATA_DIR, "combined.csv")
    df.to_csv(CSV_PATH, index=False)

print(f"[+] Dataset: {CSV_PATH}")

# ============================================================
# 2. LOAD + PROFILE
# ============================================================
df = pd.read_csv(CSV_PATH, low_memory=False)

if LABEL_COLUMN not in df.columns:
    raise ValueError(
        f"'{LABEL_COLUMN}' not found. Available columns: {list(df.columns)}"
    )

print(f"[+] Shape: {df.shape}")
print("[+] Classes:")
print(df[LABEL_COLUMN].value_counts())

# ============================================================
# 3. CLEAN
# ============================================================
df = df.replace([np.inf, -np.inf], np.nan)
df = df.drop_duplicates()

X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN]

# Remove obvious provenance columns
DROP_COLUMNS = [
    "_source_file",
    "_source_class",
    "source_file",
    "source_class",
    "filename",
    "file_name"
]

X = X.drop(
    columns=[c for c in DROP_COLUMNS if c in X.columns],
    errors="ignore"
)

# ============================================================
# 4. TRAIN / TEST SPLIT
# Test set remains untouched until final evaluation.
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

# ============================================================
# 5. PREPROCESS
# EBM handles categorical variables natively.
# Imputation statistics are learned from TRAIN only.
# ============================================================
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

# Numeric: median from training data
for col in numeric_cols:
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

# Categorical: explicit missing category
for col in categorical_cols:
    X_train[col] = X_train[col].astype(str).fillna("MISSING")
    X_test[col] = X_test[col].astype(str).fillna("MISSING")

# ============================================================
# 6. OPTUNA OBJECTIVE
# No SMOTE before CV.
# Each CV validation fold remains untouched.
# ============================================================
def objective(trial):
    params = {
        "max_bins": trial.suggest_int(
            "max_bins", 32, 256, step=32
        ),
        "max_interaction_bins": trial.suggest_int(
            "max_interaction_bins", 16, 64, step=16
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.1, log=True
        ),
        "max_rounds": trial.suggest_int(
            "max_rounds", 200, 2000, step=200
        ),
        "interactions": trial.suggest_int(
            "interactions", 0, 20
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 2, 50
        )
    }

    model = ExplainableBoostingClassifier(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    cv = StratifiedKFold(
        n_splits=CV_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1
    )

    return scores.mean()

# ============================================================
# 7. OPTUNA OPTIMIZATION
# ============================================================
study = optuna.create_study(
    direction="maximize",
    study_name="ForenXAI_EBM"
)

study.optimize(
    objective,
    n_trials=OPTUNA_TRIALS
)

print(
    f"[+] Best CV F1-macro: "
    f"{study.best_value:.4f}"
)

print(
    f"[+] Best parameters: "
    f"{study.best_params}"
)

# ============================================================
# 8. FINAL EBM TRAINING
# Uses TRAIN only.
# ============================================================
ebm = ExplainableBoostingClassifier(
    **study.best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

ebm.fit(
    X_train,
    y_train
)

# ============================================================
# 9. FINAL TEST EVALUATION
# Test set is used ONCE.
# ============================================================
y_pred = ebm.predict(X_test)
y_proba = ebm.predict_proba(X_test)[:, 1]

print("\n=== FINAL TEST RESULTS ===")
print(classification_report(y_test, y_pred))

print(
    f"F1-macro: "
    f"{f1_score(y_test, y_pred, average='macro'):.4f}"
)

print(
    f"ROC-AUC: "
    f"{roc_auc_score(y_test, y_proba):.4f}"
)

print(
    f"PR-AUC: "
    f"{average_precision_score(y_test, y_proba):.4f}"
)

# ============================================================
# 10. SAVE MODEL
# ============================================================
joblib.dump(
    ebm,
    os.path.join(
        OUTPUT_DIR,
        "forenxai_ebm.pkl"
    )
)

# ============================================================
# 11. SAVE GLOBAL EXPLANATION
# EBM-native explanation.
# ============================================================
global_explanation = ebm.explain_global(
    name="ForenXAI EBM Global Explanation"
)

joblib.dump(
    global_explanation,
    os.path.join(
        OUTPUT_DIR,
        "ebm_global_explanation.pkl"
    )
)

# ============================================================
# 12. SAVE LOCAL EXPLANATION
# ============================================================
local_explanation = ebm.explain_local(
    X_test.iloc[:100],
    y_test.iloc[:100],
    name="ForenXAI EBM Local Explanation"
)

joblib.dump(
    local_explanation,
    os.path.join(
        OUTPUT_DIR,
        "ebm_local_explanation.pkl"
    )
)

# ============================================================
# 13. SAVE METADATA
# ============================================================
metadata = {
    "model": "ExplainableBoostingClassifier",
    "dataset": "UWF-ZeekData24",
    "label_column": LABEL_COLUMN,
    "random_state": RANDOM_STATE,
    "cv_folds": CV_SPLITS,
    "optuna_trials": OPTUNA_TRIALS,
    "best_cv_f1_macro": float(study.best_value),
    "best_parameters": study.best_params,
    "test_f1_macro": float(
        f1_score(
            y_test,
            y_pred,
            average="macro"
        )
    ),
    "test_roc_auc": float(
        roc_auc_score(
            y_test,
            y_proba
        )
    ),
    "features": list(X.columns),
    "categorical_features": categorical_cols,
    "numeric_features": numeric_cols
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "metadata.json"
    ),
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print(
    f"\n[+] EBM pipeline completed."
)

print(
    f"[+] Artifacts saved to: "
    f"{OUTPUT_DIR}/"
)

Retrieving folder contents


Processing file 1eLfFe6juUdwaJkZyhoFBtHgUhvlu0Pmd UWF_ZeekData24_combined.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1eLfFe6juUdwaJkZyhoFBtHgUhvlu0Pmd
To: /content/raw_data/UWF_ZeekData24_combined.csv
100%|██████████| 29.9M/29.9M [00:00<00:00, 35.5MB/s]
Download completed


[+] Dataset: raw_data/UWF_ZeekData24_combined.csv
[+] Shape: (95871, 29)
[+] Classes:
label_binary
False        47906
True         46995
Duplicate      970
Name: count, dtype: int64


[I 2026-07-21 12:48:47,733] A new study created in memory with name: ForenXAI_EBM
